# Wavefront-Curvature Phase Shift with aispy

This notebook reproduces Figures 9 and 10 from
[Mouelle et al. (2025)](https://arxiv.org/abs/2510.26739),
demonstrating the full **aispy** workflow for a
Phase-Shear Readout (PSR) Mach–Zehnder atom interferometer with LMT-101 beam splitters.

The workflow has three stages:

1. **Build inputs** — use `aispy.utils.AISFlow` to generate `.aisi` input files
   for a 5 × 5 grid of initial cloud positions and velocities (25 simulations).
2. **Run simulations** — call `ais++` once per input file to produce HDF5 outputs.
3. **Analyse** — load positions and fit PSR fringe patterns (**Fig. 9**),
   then compare extracted phase and fringe wave-vector against the analytical
   wavefront-curvature model (**Fig. 10**).

Pre-generated output files are included so you can run the analysis cells
immediately without building `ais++`.

---

**Quick start**

```bash
# 0. Install dependencies
pip install aispy numpy scipy matplotlib h5py pandas jupyter

# 1. (Optional) Generate ais++ input files
cd examples/
python build_inputs.py --natoms 100000 --nlmt 101

# 2. (Optional) Run simulations  — one per input file
#    (each takes ~2 min on a modern laptop; pre-generated outputs are included)
for f in input-files/PSR_WA_NLMT101_*.aisi; do
    stem=$(basename "$f" .aisi)
    ais++ -i "$f" -o "output-files/${stem}.h5"
done

# 3. Open and run this notebook
jupyter notebook example_1.ipynb
```

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import os
import h5py
from scipy.optimize import curve_fit
from cycler import cycler

# ── publication style ────────────────────────────────────────────────────────
try:
    plt.style.use('seaborn-v0_8-ticks')
except OSError:
    plt.style.use('seaborn-ticks')

mpl.rcParams.update({
    'font.family'        : 'serif',
    'font.serif'         : ['Times New Roman', 'Times', 'Palatino'],
    'text.usetex'        : True,
    'text.latex.preamble': r'\usepackage{times}',
    'font.size'          : 8,
    'axes.titlesize'     : 8,
    'axes.labelsize'     : 8,
    'lines.linewidth'    : 1.0,
    'lines.markersize'   : 4,
    'xtick.labelsize'    : 7,
    'ytick.labelsize'    : 7,
    'xtick.direction'    : 'in',
    'ytick.direction'    : 'in',
    'xtick.major.size'   : 3,
    'ytick.major.size'   : 3,
    'legend.fontsize'    : 6.5,
    'legend.frameon'     : False,
    'axes.grid'          : False,
    'savefig.dpi'        : 300,
    'savefig.bbox'       : 'tight',
})
mpl.rcParams['axes.prop_cycle'] = cycler(
    color=['#1b9e77', '#d95f02', '#7570b3', '#e7298a', '#66a61e'])

## 1. Building inputs with `AISFlow`

`aispy.utils.AISFlow` translates a Python parameter dictionary into a
ready-to-run `.aisi` file.  The cell below shows the recipe for a
**single** PSR Mach–Zehnder simulation ($n = 101$, $x_0 = 0$, $v_{x0} = 0$).
The companion script `build_inputs.py` loops over the full 5 × 5 grid and
writes all 25 input files in one call.

```bash
python build_inputs.py --natoms 100000 --nlmt 101
```

In [ ]:
import sys
sys.path.insert(0, '..')
import mpmath as mp
from aispy.utils import AISFlow, pi, hbar, kz

# ── interferometer constants ──────────────────────────────────────────────────
T_interp   = mp.mpf('2.225')            # s  (half-interrogation time)
RABI_FREQ  = 2 * pi * mp.mpf('1e3')    # rad/s
ZR         = 450.085                    # m  (Rayleigh range, w0 = 3 cm)
T_detect   = 2 * T_interp + mp.mpf('0.001')

param_dict = {
    'cloud_params': {
        'natoms':       100_000,
        'initialstate': 0,
        'sigma':        100e-6,    # transverse cloud radius (m)
        'longtemp':     0,
        'transtemp':    1e-9,      # 1 nK
        'x0':           [0.0, 0.0, 0.0],
        'v0':           [0.0, 0.0, 19.62],
    },
    'potential_params': {'utype': 'linear_pot'},
    'sequence_params': {
        't_init':            mp.mpf('0.0'),
        'detectiontime':     T_detect,
        'interrogation_time': [T_interp],
        'lmt_order':         101,
        'dt_lmt':            mp.mpf('1e-7'),
        'automaticdetuning': 1,
        'frequencychirp':    0,
        'kchirp':            0,
        'ultranarrow':       True,
        'sequencename':      'MZ',
        'loopnumber':        1,
    },
    'pulse_params': {
        'rabi_freq':      RABI_FREQ,
        'wtype':          'gaussian',
        'phi0':           0,
        'kx_psr':         3140,    # rad/m  (Phase-Shear Readout momentum kick)
        'ky_psr':         0,
        'ky':             0,
        'waist':          mp.sqrt(2 * ZR / kz),
        'focallength':    0,
        'zupwardlaser':   0,
        'zdownwardlaser': 0,
        'beam_radius':    0.02,
        'baseline':       10,
        'zernike_params': {'1': [0,0,0,0,0,0], '4': [0,0,0,0,0,0]},
    },
    'simulation_params': {
        'amplitudethreshold': 0,
        'coherencelength':    10 * 0.5 * float(hbar) / (1.44e-25 * 1.38e-23 * 10e-9)**0.5,
        'usemcbranching':     0,
        'ignoredetuning':     0,
        'usestaticapprox':    0,
        'seed':               -1,
        'usedetvolselection': 0,
        'usepathselection':   1,
        'xdet':               [-3e-2, 3e-2],
        'ydet':               [-3e-2, 3e-2],
        'zdet':               [-3e-2, 3e-2],
        'gslqagabserr':       1e-12,
        'gslqagrelerr':       1e-12,
        'gslkinodeabserr':    1e-9,
        'gslkinoderelerr':    0,
        'gslpulseodeabserr':  1e-9,
        'gslpulseoderelerr':  0,
        'ultrafast':          1,
    },
    'io_params': {'printprobs': 0, 'printwavepackets': 0},
}

# Write the .aisi file — AISFlow(param_dict, stem, output_directory)
os.makedirs('input-files', exist_ok=True)
AISFlow(param_dict, 'PSR_WA_NLMT101_VX00.000e+00_X00.000e+00', 'input-files')
print('Written: input-files/PSR_WA_NLMT101_VX00.000e+00_X00.000e+00.aisi')

## 2. Load pre-generated data

Running all 25 simulations at $N = 10^5$ atoms each takes roughly an hour
on a modern laptop.  Pre-generated outputs are included in `data/` and
`output-files/` so you can explore the analysis immediately.

**CSV files** (`data/`) contain the PSR fringe-fit parameters
($\kappa$, $\varphi_0$, $C$, $\sigma$, $\mu_x$) for all 25 simulations,
one CSV per initial velocity $v_{x0}$.  
**HDF5 files** (`output-files/`) contain the raw per-atom positions and
states for the $v_{x0} = 0$ column (used for Fig. 9).

In [ ]:
# ── grid of initial conditions ────────────────────────────────────────────────
vx0_list = [-2e-4, -1e-4, 0.0, 1e-4, 2e-4]   # m/s
x0_list  = [-1e-3, -5e-4,  0.0, 5e-4,  1e-3]  # m

# color palette — one paired shade per vx0 column
colors = [
    '#1b9e77', '#a6dba0',
    '#d95f02', '#fdb863',
    '#7570b3', '#c2a5cf',
    '#e7298a', '#fbb4b9',
    '#66a61e', '#b8e186',
]

# ── load CSV fit-parameter tables ─────────────────────────────────────────────
DATA_DIR = 'data'
NATOMS, NLMT, NSHOTS = int(1e5), 101, 1

csv_data = {}
pattern  = f'EXPERIMENT3_NATOMS{NATOMS:.3e}_NLMT{NLMT}_F0.000e+00ZR_NSHOTS{NSHOTS}_VX0'
for fname in os.listdir(DATA_DIR):
    if fname.startswith(pattern) and fname.endswith('.csv'):
        vx0 = float(fname.split('VX0')[1].replace('.csv', ''))
        df  = pd.read_csv(os.path.join(DATA_DIR, fname))
        df.columns = df.columns.str.strip()
        csv_data[vx0] = df.sort_values('x0').reset_index(drop=True)

vx0_sorted = sorted(csv_data.keys())
print(f'Loaded {len(csv_data)} CSV files  ({[f"{v*1e3:.1f}" for v in vx0_sorted]} mm/s)')

# ── interferometer physics ────────────────────────────────────────────────────
k    = 8995954.05   # rad/m  (698 nm Sr clock transition)
T    = 2.225        # s      (free-evolution time)
tau  = 0.5e-3       # s      (pulse duration)
zR   = 450.085      # m      (Rayleigh range, w0 = 3 cm)
vz0  = 19.62        # m/s    (launch velocity)
g_   = 9.81         # m/s²
n    = NLMT
tdet = 2 * T        # s      (detection time after launch)
Teff = np.sqrt(T * (T - n * tau))

# transverse cloud spreads at detection
sig_x0  = 1e-4          # m    (initial position spread)
sig_vx0 = 3.1e-4        # m/s  (≈ 1 nK)
sig_x_det = np.sqrt(sig_x0**2 + (sig_vx0 * tdet)**2)
cexp = sig_vx0**2 * tdet / sig_x_det**2   # 1/s

# ── analytical wavefront-curvature model (Table I & II, Mouelle et al. 2025) ──
f_abs, z0 = 0.0, 0.0

def _c1(sign):
    fz = f_abs + sign * z0
    return sign * k * (zR**2 - fz**2) / (2 * (zR**2 + fz**2)**2)

def _c0(sign):
    fz = f_abs + sign * z0
    return k * (f_abs*zR**2 + fz**2*(f_abs + sign*2*z0)) / (2*(zR**2 + fz**2)**2)

c1p, c1m = _c1(+1), _c1(-1)
c0p, c0m = _c0(+1), _c0(-1)
denom_c  = c1p*(n+1) - c1m*(n-1)
C0       = 0.5 * Teff**2 * denom_c

Cxx0 = -g_
Cxv0 = 4*vz0 - 6*g_*T
Cvv0 = (2*(c0p*(n+1) - c0m*(n-1)) / denom_c
        - (7*g_*T**2 - 6*T*vz0 - 2*z0))
Cxx  = Cxx0
Cxv  = Cxv0 - 2*Cxx0*tdet
Cvv  = Cvv0 - Cxv0*tdet + Cxx0*tdet**2

def model_dphi(mu_x0, mu_vx0):
    """Analytical ∆φ - ∆φ₀  [rad]  (Eq. 39, Mouelle et al. 2025)."""
    mu_x = mu_x0 + mu_vx0 * tdet
    b0   = C0 * Cvv * (mu_vx0 - mu_x * cexp)**2
    b2   = C0 * (Cxx + cexp * (Cvv * cexp + Cxv))
    return b0 - b2 * mu_x**2

def model_kappa(mu_x0, mu_vx0):
    """Analytical κ - κ₀  [rad/m]  (Eq. 39, Mouelle et al. 2025)."""
    mu_x = mu_x0 + mu_vx0 * tdet
    b1   = C0 * (2*Cvv*cexp + Cxv) * (mu_vx0 - mu_x * cexp)
    b2   = C0 * (Cxx + cexp * (Cvv * cexp + Cxv))
    return b1 + 2*b2 * mu_x

# ── baselines at (x0=0, vx0=0) ────────────────────────────────────────────────
df0    = csv_data[0.0]
idx0   = (df0['x0'] - 0).abs().idxmin()
phi0_0 = df0.loc[idx0, 'phi0']
kap0   = df0.loc[idx0, 'kappa']

print(f'Baseline  φ₀ = {phi0_0*1e3:.1f} mrad')
print(f'Baseline  κ₀ = {kap0:.0f} rad/m')

## 3. Figure 9 — 2D atom clouds and PSR fringe patterns

For $v_{x0} = 0$ and five initial cloud positions
$\mu_{x_0} \in \{-1, -0.5, 0, 0.5, 1\}\,\text{mm}$:

- **Top row**: 2D density histograms of ground-state atoms at detection time.
  The PSR momentum kick $\hbar\kappa_\mathrm{PSR}$ imprints a
  grating-like spatial modulation.
- **Bottom row**: $x$-projection with the fitted
  $N(x) = A[1 + C\cos(\kappa x + \varphi_0)]\,e^{-(x-\mu_x)^2/2\sigma^2}$ model
  (pink) and Gaussian envelope (dashed).

The position-dependent fringe wave-vector $\kappa$ encodes the
wavefront-curvature aberration.

In [ ]:
# ── fringe model ──────────────────────────────────────────────────────────────
def fringe_model(x, A, sigma, mux, kappa, phi0, C):
    return A * (1 + C*np.cos(kappa*x + phi0)) * np.exp(-(x-mux)**2 / (2*sigma**2))

def gauss_env(x, A, sigma, mux):
    return A * np.exp(-(x-mux)**2 / (2*sigma**2))

# ── load h5 files and fit fringes (vx0 = 0, five x0 values) ──────────────────
OUTPUT_DIR = 'output-files'
x0_sorted  = sorted(x0_list)
fig9_data  = {}

for x0 in x0_sorted:
    fname = f'X0{x0:.3e}_COMBINED.h5'
    fpath = os.path.join(OUTPUT_DIR, fname)

    with h5py.File(fpath, 'r') as f:
        pos   = f['positions'][:]       # (N, 3): x, y, z
        states = f['states'][:]         # 0 = ground, 1 = excited
        iflag  = f['interferingFlag'][:]

    mask = (states == 0) & (iflag == 1)
    x_g  = pos[mask, 0]
    y_g  = pos[mask, 1]

    # fit 1D histogram
    counts, edges = np.histogram(x_g, bins=100)
    xc  = (edges[:-1] + edges[1:]) / 2
    err = np.where(counts > 0, np.sqrt(counts), np.inf)

    try:
        popt, pcov = curve_fit(
            fringe_model, xc, counts,
            p0=[counts.max(), np.std(x_g), x0, kap0, 0.0, 0.8],
            sigma=err, maxfev=5000)
    except RuntimeError:
        popt = np.array([counts.max(), np.std(x_g), x0, kap0, 0.0, 0.8])

    fig9_data[x0] = dict(x_g=x_g, y_g=y_g, popt=popt, counts=counts, xc=xc)
    print(f'x0={x0*1e3:+.1f} mm  N_g={mask.sum():5d}  '
          f'κ={popt[3]:.0f} rad/m  C={popt[5]:.3f}  '
          f'σ={popt[1]*1e3:.2f} mm')

# ── plot Figure 9 ─────────────────────────────────────────────────────────────
C_BAR = '#4daf4a'
C_FIT = '#f4a3b4'
C_ENV = '#f7b6c2'
NBINS = 100

xlabels = [r'$\mu_{{x_0}}={:+.1f}$ mm'.format(x0*1e3) for x0 in x0_sorted]

fig, axs = plt.subplots(
    2, 5, figsize=(7.5, 3.25), dpi=300,
    gridspec_kw=dict(hspace=0.15, wspace=0.20, height_ratios=[3, 1]),
    sharex='col',
)

ymin1, ymax1 = np.inf, -np.inf
ymin2, ymax2 = np.inf, -np.inf

for col, x0 in enumerate(x0_sorted):
    d      = fig9_data[x0]
    x_g    = d['x_g']; y_g = d['y_g']
    popt   = d['popt']; counts = d['counts']; xc = d['xc']

    # top: 2D density
    ax2d = axs[0, col]
    ax2d.set_facecolor(mpl.colormaps['jet'](0.0))
    ax2d.hist2d(x_g*1e3, y_g*1e3, bins=NBINS, cmap='jet')
    ax2d.set_title(xlabels[col], fontsize=7)
    if col == 0:
        ax2d.set_ylabel(r'$y$ [mm]', fontsize=7)
    else:
        ax2d.set_yticklabels([])
    ax2d.tick_params(labelsize=6)

    # bottom: histogram + fit
    ax1d = axs[1, col]
    bw   = (xc[1] - xc[0]) * 1e3
    bars = ax1d.bar(xc*1e3, counts, width=bw, color=C_BAR, alpha=0.75,
                    linewidth=0, label='Monte Carlo simulation')

    xfit = np.linspace(x_g.min(), x_g.max(), 600)
    fit_l, = ax1d.plot(xfit*1e3, fringe_model(xfit, *popt),
                       color=C_FIT, lw=1.3, label='Fitted model')
    env_l, = ax1d.plot(xfit*1e3, 2*gauss_env(xfit, popt[0], popt[1], popt[2]),
                       color=C_ENV, lw=1.1, ls='--', label='Gaussian envelope')

    ax1d.set_xlim(-6, 6)
    ax1d.set_ylim(0, None)
    ax1d.set_xlabel(r'$x$ [mm]', fontsize=7)
    if col == 0:
        ax1d.set_ylabel('Counts', fontsize=7)
        legend_handles = [bars, fit_l, env_l]
        legend_labels  = ['Monte Carlo simulation', 'Fitted model', 'Gaussian envelope']
    else:
        ax1d.set_yticklabels([])
    ax1d.tick_params(labelsize=6)

    y0, y1 = ax2d.get_ylim(); ymin1 = min(ymin1, y0); ymax1 = max(ymax1, y1)
    y0, y1 = ax1d.get_ylim(); ymin2 = min(ymin2, y0); ymax2 = max(ymax2, y1)

for col in range(5):
    axs[0, col].set_ylim(ymin1, ymax1)
    axs[1, col].set_ylim(ymin2, ymax2)

fig.subplots_adjust(bottom=0.18)
fig.legend(legend_handles, legend_labels,
           loc='lower center', ncol=3, frameon=False,
           fontsize=6.5, bbox_to_anchor=(0.5, 0.01))

fig.savefig('fig9_psr_atom_clouds.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig9_psr_atom_clouds.png')

## 4. Figure 10 — Position-resolved fringe parameters vs initial COM

Fitting the PSR fringe pattern for each of the 25 simulations yields
$\hat{\varphi}_0$ and $\hat{\kappa}$ as functions of the initial cloud
centre-of-mass $(\mu_{x_0}, \mu_{v_{x0}})$.
The dashed curves are the analytical wavefront-curvature model from
[Mouelle et al. (2025)](https://arxiv.org/abs/2510.26739), Eq. (39).

Each column corresponds to one initial transverse velocity $\mu_{v_{x0}}$.
The good agreement between simulation and theory across the full grid
validates both the numerical model and the analytical expansion.

In [ ]:
x0_smooth = np.linspace(-1.2e-3, 1.2e-3, 300)

fig, axs = plt.subplots(
    2, 5, figsize=(10, 3.8), dpi=150,
    gridspec_kw={'hspace': 0.55, 'wspace': 0.38},
)

for col, vx0 in enumerate(vx0_sorted):
    ax_phi   = axs[0, col]
    ax_kappa = axs[1, col]
    df       = csv_data[vx0]
    c        = colors[2 * col]   # one color per column

    dphi_data   = (df['phi0']  - phi0_0).values * 1e3   # mrad
    dkappa_data = (df['kappa'] - kap0  ).values          # rad/m

    # MC data with 2σ error bars
    ax_phi.errorbar(
        df['x0']*1e3, dphi_data,
        yerr=2e3 * df['std_phi0'].values,
        fmt='o', capsize=3, color=c, markersize=4,
        label=r'MC data (95\% C.I.)')
    ax_kappa.errorbar(
        df['x0']*1e3, dkappa_data,
        yerr=2 * df['std_kappa'].values,
        fmt='o', capsize=3, color=c, markersize=4)

    # analytical model
    ax_phi.plot(
        x0_smooth*1e3, model_dphi(x0_smooth, vx0)*1e3,
        '--', color=c, lw=1.2, label='Analytical model')
    ax_kappa.plot(
        x0_smooth*1e3, model_kappa(x0_smooth, vx0),
        '--', color=c, lw=1.2)

    ax_phi.set_title(
        r'$\mu_{{v_{{x0}}}}=$' + f' {vx0*1e3:.1f}' + r' mm\,s$^{-1}$',
        fontsize=7)

    for ax in [ax_phi, ax_kappa]:
        ax.axhline(0, color='gray', lw=0.5, ls=':')
        ax.set_xlabel(r'$\mu_{x_0}$ [mm]', fontsize=7)
        ax.tick_params(labelsize=6)

    if col == 0:
        ax_phi.set_ylabel(
            r'$\hat{\Delta\varphi} - \Delta\varphi_0$ [mrad]', fontsize=7)
        ax_kappa.set_ylabel(
            r'$\hat{\kappa} - \kappa_0$ [mrad\,mm$^{-1}$]', fontsize=7)
        ax_phi.legend(fontsize=5.5, loc='upper left')

fig.savefig('fig10_phase_parameters.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: fig10_phase_parameters.png')